In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize
from imageio.v2 import imread
import matplotlib.lines as mlines
from matplotlib.legend_handler import HandlerLine2D
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, Markdown

plt.rcParams['figure.subplot.left'] = 0


class Sandpit:
    def __init__(self, f):
        # Default options
        self.game_mode = 0  # 0 - Jacobian, 1 - Depth Only, 2 - Steepest Descent
        self.grad_length = 1/5
        self.grad_max_length = 1
        self.arrowhead_width = 0.1
        self.arrow_placement = 2  # 0 - tip, 1 - base, 2 - centre, 3 - tail
        self.tol = 0.15
        self.markerColour = (1, 0.85, 0)

        self.contourCM = LinearSegmentedColormap.from_list("Cmap", [
            (0., 0.00505074, 0.191104),
            (0.155556, 0.0777596, 0.166931),
            (0.311111, 0.150468, 0.142758),
            (0.466667, 0.223177, 0.118585),
            (0.622222, 0.295886, 0.094412),
            (0.777778, 0.368595, 0.070239),
            (0.822222, 0.389369, 0.0633324),
            (0.866667, 0.410143, 0.0564258),
            (0.911111, 0.430917, 0.0495193),
            (0.955556, 0.451691, 0.0426127),
            (1., 0.472465, 0.0357061)
        ], N=256)

        self.start_text = "**Click anywhere in the sandpit to place the dip-stick.**"
        self.win_text = "### Congratulations!\nWell done, you found the phone."

        self.revealed = False
        self.handler_map = {}
        self.nGuess = 0
        self.msgbox = widgets.Output()

        self.original_f = f

        # Find global min & max
        x0 = self.x0 = differential_evolution(
            lambda xs: f(xs[0], xs[1]), ((0, 6), (0, 6))
        ).x
        x1 = differential_evolution(
            lambda xs: -f(xs[0], xs[1]), ((0, 6), (0, 6))
        ).x

        f0 = f(x0[0], x0[1])
        f1 = f(x1[0], x1[1])

        # Normalize function
        self.f = lambda x, y: 8 * (f(x, y) - f1) / (f1 - f0) - 1

        # Numerical gradient
        self.df = lambda x, y: np.array([
            self.f(x+0.01, y) - self.f(x-0.01, y),
            self.f(x, y+0.01) - self.f(x, y-0.01)
        ]) / 0.02

        # Numerical Hessian
        self.d2f = lambda x, y: np.array([
            [
                self.df(x+0.01, y)[0] - self.df(x-0.01, y)[0],
                self.df(x, y+0.01)[0] - self.df(x, y-0.01)[0]
            ],
            [
                self.df(x+0.01, y)[1] - self.df(x-0.01, y)[1],
                self.df(x, y+0.01)[1] - self.df(x, y-0.01)[1]
            ]
        ]) / 0.02

    def draw(self):
        self.fig, self.ax = plt.subplots()
        self.ax.set_xlim([0, 6])
        self.ax.set_ylim([0, 6])
        self.ax.set_aspect(1)

        self.fig.canvas.mpl_connect('button_press_event', self.onclick)
        self.drawcid = self.fig.canvas.mpl_connect('draw_event', self.ondraw)

        self.leg = self.ax.legend(handles=[], bbox_to_anchor=(1.05, 1),
                                  loc=2, borderaxespad=0., title="Depths:")

        img = imread("readonly/sand.png")
        self.ax.imshow(img, zorder=0, extent=[0, 6, 0, 6],
                       interpolation="bilinear")

        display(self.msgbox)

    def onclick(self, event):
        if event.button != 1 or event.xdata is None:
            return

        x, y = event.xdata, event.ydata
        self.placeArrow(x, y)

        if np.linalg.norm(self.x0 - [x, y]) <= self.tol:
            self.showContours()
            return

        lx = minimize(lambda xs: self.f(xs[0], xs[1]),
                      np.array([x, y])).x

        if np.linalg.norm(lx - [x, y]) <= self.tol:
            self.local_min(lx[0], lx[1])

    def ondraw(self, event):
        self.fig.canvas.mpl_disconnect(self.drawcid)
        self.displayMsg(self.start_text)

    def placeArrow(self, x, y, auto=False):
        d = -self.df(x, y) * self.grad_length
        norm = np.linalg.norm(d)
        if norm == 0:
            return

        dhat = d / norm
        d = d * np.clip(norm, 0, self.grad_max_length) / norm

        off = d / 2

        if auto:
            self.ax.plot([x], [y], 'yo', zorder=25,
                         color="red", ms=6)
        else:
            self.nGuess += 1
            p, = self.ax.plot([x], [y], 'yo', zorder=25,
                              label=f"{self.nGuess}) {self.f(x,y):.2f}m",
                              color=self.markerColour,
                              ms=8, markeredgecolor="black")

            if self.nGuess <= 25:
                self.handler_map[p] = HandlerLine2D(numpoints=1)
                self.leg = self.ax.legend(
                    handler_map=self.handler_map,
                    bbox_to_anchor=(1.05, 1),
                    loc=2, borderaxespad=0.,
                    title="Depths:"
                )
            elif not self.revealed:
                self.showContours()
                self.displayMsg("**Too many tries! Reload and try again.**")

        self.ax.arrow(x-off[0], y-off[1], d[0], d[1],
                      linewidth=1.5,
                      head_width=self.arrowhead_width,
                      head_starts_at_zero=False,
                      zorder=20,
                      color="black")

    def showContours(self):
        if self.revealed:
            return

        x0 = self.x0
        X, Y = np.meshgrid(np.arange(0, 6, 0.05),
                           np.arange(0, 6, 0.05))

        self.ax.contour(X, Y, self.f(X, Y), 10,
                        cmap=self.contourCM)

        img = imread("readonly/phone2.png")
        self.ax.imshow(img, zorder=30,
                       extent=[x0[0]-0.1875, x0[0]+0.1875,
                               x0[1]-0.1875, x0[1]+0.1875],
                       interpolation="bilinear")

        self.displayMsg(self.win_text)
        self.revealed = True

    def local_min(self, x, y):
        img = imread("readonly/nophone.png")
        self.ax.imshow(img, zorder=30,
                       extent=[x-0.1875, x+0.1875,
                               y-0.1875, y+0.1875],
                       interpolation="bilinear")

        if not self.revealed:
            self.displayMsg(
                "**Oh no!** You've got stuck in a local optimum. Try somewhere else!"
            )

    def displayMsg(self, msg):
        self.msgbox.clear_output()
        with self.msgbox:
            display(Markdown(msg))